In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import snapatac2 as snap
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
import pyranges as pr
import os
import warnings
import requests
# plt.rcdefaults()

# import sys
# sys.path.append('/scratch/eli')
# from perturbseq import *

In [ ]:
# relies on output of cropseq_call_guides_multiome.py and cellranger output

def cut_qc_metrics_by_called(sample_name, figsize = (15,5), mode = 'gex'):

    id_files = [file for file in os.scandir("/data/norman/eli/T7/202404_SIRLOIN_multiome/guide_calling/T7_outs") if "called_ids" in file.name]
    called_ids = pd.read_csv([f for f in id_files if  sample_name in f.name][0]).reset_index().groupby("CB").UMI.count().to_frame("n_guides")
    called_ids.index = called_ids.index.map(lambda cb: cb + "-1")

    if mode == 'gex':

        adata_files = [file for file in os.scandir("/data/norman/eli/T7/202404_SIRLOIN_multiome/") if sample_name in file.name]
        adata_path = adata_files[0].path + '/outs/filtered_feature_bc_matrix.h5'

        adata = sc.read_10x_h5(adata_path)
        adata.var_names_make_unique()
        adata.var["mito"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, inplace=True, qc_vars = ["mito"])
        
        merge = adata.obs.join(called_ids, how = 'left')
        merge["called"] = merge["n_guides"].fillna(0).map(lambda n: n > 0)
        merge.drop_duplicates(inplace = True)

        fig, ax = plt.subplots(1, 3, figsize=figsize)
        sns.histplot(data = merge, x = 'log1p_total_counts', kde = True, hue = 'called', ax = ax[0])
        sns.histplot(data = merge, x = 'log1p_n_genes_by_counts', kde = True, hue = 'called', ax = ax[1])
        sns.histplot(data = merge, x = 'pct_counts_mito', kde = True, hue = 'called', ax = ax[2])
        ax[2].set_xlim([-2,50])
        fig.suptitle(f"{sample_name} GEX QC metrics by guide call")
        fig.show()

    elif mode == 'atac':

        adata_files = [file for file in os.scandir("/data/norman/eli/T7/202404_SIRLOIN_multiome/figs/intermediate_files") if sample_name in file.name and 'preprocessed_atac' in file.name]
        adata = snap.read(adata_files[0].path, backed = None)
        merge = adata.obs.join(called_ids, how = 'left')
        merge["called"] = merge["n_guides"].fillna(0).map(lambda n: n > 0)
        merge.drop_duplicates(inplace = True)

        fig, ax = plt.subplots(1, 2, figsize=figsize)
        sns.histplot(data = merge, x = 'n_fragment', kde = True, hue = 'called', ax = ax[0])
        ax[0].set_xlim([0,1e5])
        sns.histplot(data = merge, x = 'tsse', kde = True, hue = 'called', ax = ax[1])
        fig.suptitle(f"{sample_name} ATAC QC metrics by guide call")
        fig.show()

    return merge

In [ ]:
gexqc_040 = cut_qc_metrics_by_called("Lane1_040")
atacqc_040 = cut_qc_metrics_by_called("Lane1_040", mode = 'atac')
qc040 = gexqc_040.join(atacqc_040.drop(["n_guides", "called"], axis = 1), how = 'inner')
qc040.n_fragment = qc040.n_fragment.astype(int)
singlets_040 = qc040.query("n_guides == 1 and log1p_total_counts > 6 and log1p_n_genes_by_counts > 6 and n_fragment > 1000")

In [ ]:
print(len(singlets_040))
print(np.expm1(singlets_040.log1p_n_genes_by_counts).mean())
print(singlets_040.n_fragment.mean())

In [ ]:
ids_040 = pd.read_csv("/data/norman/eli/T7/202404_SIRLOIN_multiome/guide_calling/T7_outs/Lane1_040_called_ids.csv")
ids_040['n_guides'] = ids_040.groupby('CB').UMI.transform('count')
ids_040 = ids_040.query('n_guides == 1')
ids_040['guide_target'] = ids_040['identity'].map(lambda s: s.split("_")[0])
ids_040['CB'] = ids_040.CB.map(lambda b: b + "-1")
ids_040.set_index('CB', inplace = True)

In [ ]:
singlets_040 = singlets_040.join(ids_040['guide_target'], how = 'left')

In [ ]:
# direct output of cellranger

raw = sc.read_10x_h5("/data/norman/eli/T7/202404_SIRLOIN_multiome/Lane1_040/outs/filtered_feature_bc_matrix.h5")
raw.var_names_make_unique()
raw = raw[raw.obs.index.isin(singlets_040.index)]
raw.obs = raw.obs.join(singlets_040, how = 'right')
raw

In [ ]:
atac = snap.read("analysis/intermediate_files/Lane1_040_preprocessed_atac.h5ad", backed = None)
atac = atac[atac.obs.index.isin(raw.obs.index)]
atac

In [ ]:
assert np.all(atac.obs.index == raw.obs.index)

In [ ]:
atac.obs = atac.obs.join(raw.obs[['guide_target']])
atac.obs

In [ ]:
raw.write("040_singlets_gex_unprocessed.h5ad")
atac.write("040_atac_singlets_unprocessed.h5ad")

In [ ]:
raw = sc.read('/data1/normantm/eli/T7/202404_SIRLOIN_multiome/share/040_singlets_gex_unprocessed.h5ad')
raw

In [ ]:
raw.obs.to_csv("/data1/normantm/eli/T7/202404_SIRLOIN_multiome/share/singlets_assigned.csv")